# 03 — Building Characteristics (Paris / EUBUCCO)

Aggregates EUBUCCO building-level statistics per grid cell.

**Replaces:** NYC PLUTO notebook 03 with EUBUCCO open data.

**Features extracted per cell:**
- `avg_height` — mean building height (m)
- `avg_floors` — mean number of floors
- `avg_construction_year` — mean year built
- `building_count` — number of buildings in cell
- `residential_ratio` — fraction of residential buildings

**Output file:** `csv/Paris/03_building_characteristics.csv`

In [1]:
PARIS_CONFIG = "paris.json"

In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd
import json
import os
import math

with open(PARIS_CONFIG, encoding="utf-8") as f:
    config = json.load(f)

NUTS_CODE   = config["nuts_code"]
CELL_SIZE_M = config["grid_cell_size_m"]
CSV_DIR     = config["csv_dir"]
os.makedirs(CSV_DIR, exist_ok=True)

# Load grid
df_grid = pd.read_csv(f"{CSV_DIR}/01_grid_definition.csv", dtype={"cell_id": str})
valid_cells = set(df_grid["cell_id"])
print(f"Loaded {len(df_grid)} grid cells")

# Recover grid parameters saved by notebook 01
gp      = config["grid_params"]
LAT_MIN = gp["lat_min"]
LON_MIN = gp["lon_min"]
LAT_STEP = gp["lat_step"]
LON_STEP = gp["lon_step"]
print(f"Grid params loaded from paris.json")

Loaded 120331 grid cells
Grid params loaded from paris.json


In [3]:
# ── Stream EUBUCCO ────────────────────────────────────
storage_opts = {
    "anon": True,
    "client_kwargs": {"endpoint_url": "https://s3.eubucco.com"}
}
path = f"s3://eubucco/v0.2/buildings/parquet/nuts_id={NUTS_CODE}/{NUTS_CODE}.parquet"
print(f"Streaming EUBUCCO...")

gdf = gpd.read_parquet(path, storage_options=storage_opts)
gdf = gdf.to_crs("EPSG:4326")
gdf["latitude"]  = gdf.geometry.centroid.y
gdf["longitude"] = gdf.geometry.centroid.x
gdf = gdf.dropna(subset=["latitude", "longitude"])
print(f"Loaded {len(gdf):,} buildings")

Streaming EUBUCCO...


C:\Users\Hani\AppData\Local\Temp\ipykernel_18676\1953736265.py:11: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["latitude"]  = gdf.geometry.centroid.y


C:\Users\Hani\AppData\Local\Temp\ipykernel_18676\1953736265.py:12: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["longitude"] = gdf.geometry.centroid.x


Loaded 3,594,995 buildings


In [4]:
# ── Assign buildings to grid cells ────────────────────
gdf["grid_row"] = ((gdf["latitude"]  - LAT_MIN) / LAT_STEP).astype(int)
gdf["grid_col"] = ((gdf["longitude"] - LON_MIN) / LON_STEP).astype(int)
gdf["cell_id"]  = "r" + gdf["grid_row"].astype(str).str.zfill(4) + "_c" + gdf["grid_col"].astype(str).str.zfill(4)

# Keep only buildings in valid cells
gdf = gdf[gdf["cell_id"].isin(valid_cells)].copy()
print(f"Buildings in valid grid cells: {len(gdf):,}")

# Clean numeric fields
for col in ["height", "floors", "construction_year"]:
    gdf[col] = pd.to_numeric(gdf[col], errors="coerce")

gdf.loc[gdf["construction_year"] < 1000, "construction_year"] = np.nan
gdf.loc[gdf["height"] <= 0, "height"] = np.nan
gdf.loc[gdf["floors"] <= 0, "floors"] = np.nan

Buildings in valid grid cells: 3,505,711


In [5]:
# ── Aggregate per grid cell ───────────────────────────
gdf["is_residential"] = (gdf["type"] == "residential").astype(int)

agg = gdf.groupby("cell_id").agg(
    avg_height            = ("height",            "mean"),
    avg_floors            = ("floors",            "mean"),
    avg_construction_year = ("construction_year", "mean"),
    building_count        = ("cell_id",           "count"),
    residential_ratio     = ("is_residential",    "mean"),
).reset_index()

agg["avg_height"]            = agg["avg_height"].round(1)
agg["avg_floors"]            = agg["avg_floors"].round(1)
agg["avg_construction_year"] = agg["avg_construction_year"].round(0).astype("Int64")
agg["residential_ratio"]     = agg["residential_ratio"].round(3)

# Ensure all grid cells are present
df_result = df_grid[["cell_id"]].merge(agg, on="cell_id", how="left")
print(f"Aggregated {len(df_result)} cells")
print(df_result.describe().round(2).to_string())

Aggregated 120331 cells
       avg_height  avg_floors  avg_construction_year  building_count  residential_ratio
count   120331.00   120331.00               106971.0       120331.00          120331.00
mean         5.81        1.62                1952.72           29.13               0.69
std          3.32        0.96                  42.88           25.57               0.29
min          0.10        0.90                 1200.0            3.00               0.00
25%          4.10        1.20                 1934.0            9.00               0.56
50%          4.80        1.40                 1963.0           21.00               0.80
75%          6.00        1.60                 1981.0           43.00               0.90
max         93.20       24.20                 2024.0          237.00               1.00


In [6]:
# ── Save output ───────────────────────────────────────
output_path = f"{CSV_DIR}/03_building_characteristics.csv"
df_result.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_result)} rows x {df_result.shape[1]} cols)")
df_result.head(10)

Saved: csv/Paris/03_building_characteristics.csv  (120331 rows x 6 cols)


,cell_id,avg_height,avg_floors,avg_construction_year,building_count,residential_ratio
0,r0004_c0596,4.7,1.1,1880,19,0.263
1,r0004_c0597,4.4,1.4,1834,7,0.429
2,r0004_c0606,3.9,1.2,1869,22,0.591
3,r0006_c0521,4.3,1.3,1880,20,0.400
4,r0006_c0522,3.8,1.2,1998,6,0.500
5,r0006_c0601,6.2,1.1,1687,4,0.750
6,r0007_c0521,3.7,1.1,1868,19,0.158
7,r0007_c0594,3.5,1.1,1884,18,0.389
8,r0007_c0595,4.5,1.4,1918,7,0.286
9,r0007_c0596,4.4,1.3,1888,7,0.571
